In [1]:
import pandas as pd
import sqlite3
import plotly.graph_objects as go
import numpy as np

In [2]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

In [3]:
query = """
SELECT uid, timestamp, numTrials
FROM checker
WHERE uid LIKE "user_%" AND status = "ready" AND labname = "project1"
"""
commits = pd.io.sql.read_sql(query, conn, parse_dates='timestamp')
commits.sort_values('uid', inplace=True)
commits['date'] = commits.timestamp.dt.date
commits = commits.groupby(['uid', 'date']).numTrials.max().to_frame().reset_index()
commits.head()

,uid,date,numTrials
0,user_1,2020-05-14,11
1,user_10,2020-05-12,7
2,user_10,2020-05-13,21
3,user_10,2020-05-14,59
4,user_11,2020-05-03,1


In [4]:
data = commits.pivot(index='uid', columns='date', values='numTrials')
data.sort_index(axis=1, inplace=True)
data = data.ffill(axis=1).fillna(0)
data.head()


date,2020-04-17,2020-04-18,2020-04-19,2020-04-22,2020-04-23,2020-04-24,2020-05-03,2020-05-04,2020-05-05,2020-05-06,2020-05-07,2020-05-08,2020-05-09,2020-05-10,2020-05-11,2020-05-12,2020-05-13,2020-05-14,2020-05-15
uid,,,,,,,,,,,,,,,,,,,
user_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.0,11.0
user_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,21.0,59.0,59.0
user_11,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
user_12,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,4.0
user_13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,30.0,30.0,32.0,32.0


In [5]:
numOfRows = data.shape[0] # No of users
numOfCols = data.shape[1] # No of dates
numOfFrames = numOfCols
xaxis_range = [0,numOfFrames]
yaxis_range = [0, 170]

x_init = np.array([0])

initial_data = []
for curr_uid in data.index:
    initial_data.append(go.Scatter(x=x_init, y=None, mode="lines+markers", name=curr_uid))

In [6]:
# Frames
frames = []
for f in range(numOfFrames):
    x_axis = np.arange(f+1)
    curr_data = []
    for curr_uid, values in data.iterrows():
        y_axis = np.array(values.iloc[:f+1])
        curr_data.append(go.Scatter(x=x_axis, y=y_axis, mode="lines+markers", name=curr_uid))
    curr_frame = go.Frame(data=curr_data)
    frames.append(curr_frame)
 
figure = go.Figure(
    data = initial_data,
    layout = {
        "height": 600,
        "title":"Dynamic of commits per user in project1",
        "xaxis":{"range":xaxis_range, "visible":True, "showline":True, "dtick": 2},
        "yaxis":{"range":yaxis_range,"visible":True, "showline":True},
        "updatemenus":[{"type":"buttons","buttons":[{"method":"animate","label":"play", "args":[None]}]}]
        },
    frames = frames
    )
figure.show()

In [7]:
conn.close()